In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('gurgaon_properties_post_feature_selection.csv')

In [3]:
df.head()

,property_type,sector,bedRoom,bathroom,balcony,facing,built_up_area,servant room,age_category,furnishing_type,combined_rating,luxury_category,floor_category,price
0,0.0,72.0,2,2,1.0,7.0,1000.0,0,3.0,2.0,4.00,0.0,2.0,0.45
1,0.0,32.0,2,2,1.0,7.0,722.0,0,2.0,0.0,4.25,0.0,1.0,0.50
2,0.0,100.0,2,2,3.0,0.0,661.0,0,1.0,2.0,4.25,0.0,0.0,0.40
3,0.0,64.0,2,2,2.0,0.0,1333.0,0,3.0,2.0,4.20,0.0,1.0,1.47
4,0.0,96.0,2,2,3.0,0.0,1217.0,0,4.0,2.0,4.00,0.0,2.0,0.70


In [4]:
X = df.drop(columns=['price'])
y = df['price']

In [ ]:
# 1. ohe
# 2. scale and transform price (log)
# 3. model train

In [6]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

In [7]:
df.head(1)

,property_type,sector,bedRoom,bathroom,balcony,facing,built_up_area,servant room,age_category,furnishing_type,combined_rating,luxury_category,floor_category,price
0,0.0,72.0,2,2,1.0,7.0,1000.0,0,3.0,2.0,4.0,0.0,2.0,0.45


In [8]:
columns_to_encode = ['sector', 'balcony', 'age_category', 'facing','furnishing_type', 'luxury_category', 'floor_category']

In [9]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

In [13]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['property_type', 'bedRoom', 'bathroom', 'built_up_area', 'servant room', 'combined_rating']),
        ('cat', OneHotEncoder(drop='first'), columns_to_encode)
    ], 
    remainder='passthrough'
)

In [23]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    # ('regressor', LinearRegression())
    ('regressor', SVR(kernel='rbf'))

])

In [24]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [25]:
scores.mean()

np.float64(0.8833208024763713)

In [26]:
scores.std()

np.float64(0.01838979741256131)

In [27]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [28]:
pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tran

In [29]:
y_pred=pipeline.predict(X_test)

In [30]:
y_pred=np.expm1(y_pred)

In [31]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(np.expm1(y_test),y_pred)

0.5422805197743026

In [32]:
# my experimentation

In [37]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import numpy as np

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(random_state=42),
    "Lasso": Lasso(random_state=42),
    "ElasticNet": ElasticNet(random_state=42),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Extra Trees": ExtraTreesRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR(),

    # Boosting Models
    "XGBoost": XGBRegressor(
        random_state=42,
        verbosity=0
    ),

    "LightGBM": LGBMRegressor(
        random_state=42,
        verbose=-1
    ),

    "CatBoost": CatBoostRegressor(
        random_state=42,
        verbose=0
    )
}

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

print(f'{"Model":25} {"R² Score":10} {"MAE":10}')
print("-"*50)

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    y_pred_log = cross_val_predict(
        pipeline,
        X,
        y_transformed,
        cv=kfold,
        n_jobs=-1
    )

    r2 = r2_score(y_transformed, y_pred_log)

    y_true = np.expm1(y_transformed)
    y_pred = np.expm1(y_pred_log)

    mae = mean_absolute_error(y_true, y_pred)

    print(f"{name:25} {r2:.4f}     {mae:.4f}")

Model                     R² Score   MAE       
--------------------------------------------------
Linear Regression         0.8358     0.6899
Ridge                     0.8364     0.6914
Lasso                     -0.0006     1.5832
ElasticNet                -0.0006     1.5832
Decision Tree             0.7933     0.6716
Random Forest             0.8806     0.5291
Extra Trees               0.8861     0.5112
Gradient Boosting         0.8610     0.6184
AdaBoost                  0.7213     0.9281
KNN                       0.8174     0.6778
SVR                       0.8845     0.5406
XGBoost                   0.8905     0.5155
LightGBM                  0.8778     0.5595


CatBoostError: catboost/libs/train_lib/dir_helper.cpp:26: Can't create train tmp dir: tmp

In [36]:
!pip install xgboost lightgbm catboost

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.8/69.5 MB 5.6 MB/s eta 0:00:13
    --------------------------------------- 1.6/69.5 MB 4.3 MB/s eta 0:00:16
   - -------------------------------------- 2.1/69.5 MB 4.4 MB/s eta 0:00:16
   - -------------------------------------- 2.9/69.5 MB 3.8 MB/s eta 0:00:18
   -- ------------------------------------- 3.9/69.5 MB 4.0 MB/s eta 0:00:17
   -- ------------------------------------- 5.0/69.5 MB 4.1 MB/s eta 0:00:16
   --- ------------------------------------ 6.0/69.5 MB 4.2 MB/s eta 0:00:16
   ---- ----------------------------------- 7.1/69.5 MB 4.3 MB/s eta 0:00:15
   ---- ----------------------------------- 7.9/69.5 MB 4.3 MB/s eta 0:00:15
   ----- ---------------------------------- 8.9/69.5 MB 4.3 MB/s eta 0:00:15
   ----- ---------------------------------- 9.7/69.5 MB 4.2 MB/s eta 0:00:15
   ------ --------------------------------- 10.5/69.5 MB 4.2 MB/s eta 0:00:15
   --


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
# picking -  
# 1. Random RandomForest
# 2. Extra trees
# 3. SVR
# 4. xgboost

1. RANDOM FOREST

In [40]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42)

pipe = Pipeline([
    ("model", rf)
])

In [41]:
param_dist = {

    "model__n_estimators":[100,200,300,500,800,1000],

    "model__max_depth":[
        None,5,10,15,20,30,40,50
    ],

    "model__min_samples_split":[
        2,5,10,15,20
    ],

    "model__min_samples_leaf":[
        1,2,4,6,8
    ],

    "model__max_features":[
        "sqrt",
        "log2",
        0.5,
        0.7,
        None
    ],

    "model__bootstrap":[
        True,
        False
    ]
}

In [42]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(

    estimator=pipe,

    param_distributions=param_dist,

    n_iter=100,

    cv=10,

    scoring='r2',

    verbose=2,

    random_state=42,

    n_jobs=-1

)

random_search.fit(X,y_transformed)

Fitting 10 folds for each of 100 candidates, totalling 1000 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__bootstrap': [True, False], 'model__max_depth': [None, 5, ...], 'model__max_features': ['sqrt', 'log2', ...], 'model__min_samples_leaf': [1, 2, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",100
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strate

In [43]:
print(random_search.best_score_)

print(random_search.best_params_)

0.8431333213433021
{'model__n_estimators': 1000, 'model__min_samples_split': 2, 'model__min_samples_leaf': 1, 'model__max_features': 0.5, 'model__max_depth': 40, 'model__bootstrap': False}


In [ ]:
# 0.8431333213433021
# {'model__n_estimators': 1000, 'model__min_samples_split': 2, 'model__min_samples_leaf': 1, 'model__max_features': 0.5, 'model__max_depth': 40, 'model__bootstrap': False}

2. EXTRA_TREES

In [44]:
from sklearn.ensemble import ExtraTreesRegressor

pipe_extree = Pipeline([
    ("model",ExtraTreesRegressor(random_state=42))
])

In [46]:
param_dist_extree={

'model__n_estimators':[200,400,600,800,1000],

'model__max_depth':[None,10,20,30,40,50],

'model__min_samples_split':[2,5,10],

'model__min_samples_leaf':[1,2,4],

'model__max_features':[
"sqrt",
"log2",
0.5,
0.7,
None
]

}

In [47]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(

    estimator=pipe_extree,

    param_distributions=param_dist_extree,

    n_iter=100,

    cv=10,

    scoring='r2',

    verbose=10,

    random_state=42,

    n_jobs=-1

)

random_search.fit(X,y_transformed)

Fitting 10 folds for each of 100 candidates, totalling 1000 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': [None, 10, ...], 'model__max_features': ['sqrt', 'log2', ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",100
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation

In [49]:
print(random_search.best_score_)

print(random_search.best_params_)

0.8208493483303279
{'model__n_estimators': 600, 'model__min_samples_split': 2, 'model__min_samples_leaf': 1, 'model__max_features': None, 'model__max_depth': 50}
